# 03 — TIME TRAVEL & AUDIT VERIFICATION | Demo nhanh trên lớp (Task 3)

**Đề bài:**
1. Dùng PySpark `option("versionAsOf", ...)` đọc **bảng Silver trước MERGE** và so sánh với bảng sau MERGE.
2. **Audit bằng cách kiểm tra Delta History hoặc RESTORE**. Notebook chọn **History (không RESTORE live Silver)**; thêm kiểm tra `_delta_log/*.json` vì demo cuối môn yêu cầu trình bày transaction log.

## 0. Cấu hình chuẩn các mốc Task 2

Giữ nguyên `UPDATE_TRIP_ID` lấy từ script UPDATE, `INSERT_TRIP_ID` là ID mới của script INSERT. Không dùng `latest MERGE` vì mới nhất hiện tại có thể là schema v48, không phải UPDATE v45.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'silver').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'silver').is_dir():
    raise RuntimeError('Mở notebook từ thư mục gốc project hoặc thư mục notebooks/.')
sys.path.insert(0, str(ROOT))
from src.silver.silver_pipeline import make_spark

spark = make_spark('local[2]')
BRONZE = str(ROOT / 'data/bronze/taxi_trips')
SILVER = str(ROOT / 'data/silver/taxi_trips')
REJECTED = str(ROOT / 'data/silver/rejected_records')
AUDIT = str(ROOT / 'data/silver/batch_audit')

INITIAL_VERSION = 44
UPDATE_VERSION = 45
INSERT_VERSION = 47
SCHEMA_VERSION = 48
UPDATE_TRIP_ID = '93184dd0b5a98cc1efe037f60a0ab8f5f90c178bbb12809ac61b991e106c9c05'
INSERT_TRIP_ID = '33f764b320f526509a5ffc671a628fff2361a066ceeae606bb7f39abd528623f'
UPDATE_INGEST_BATCH = 'demo_20260927T080446776977'
INSERT_INGEST_BATCH = 'demo_20260927T085807873071'

def at_version(v):
    return spark.read.format('delta').option('versionAsOf', v).load(SILVER)

def trip_at(v, trip_id):
    frame = at_version(v)
    selected = [c for c in ('trip_id', 'fare_amount', 'tip_amount', 'surcharge_fee',
                            'ingest_batch_id', 'record_source', 'ingested_at') if c in frame.columns]
    rows = (frame.filter(F.col('trip_id') == trip_id)
            .select(*selected).limit(2).collect())
    return [r.asDict(recursive=True) for r in rows]

def display_comparison(rows_before, rows_after, v_before, v_after, trip_id):
    entries = []
    for label, version, rows in [('BEFORE', v_before, rows_before), ('AFTER', v_after, rows_after)]:
        if rows:
            x = rows[0]
            entries.append({'Mốc':label, 'Silver version': version,
                            'trip_id (8 ký tự)':str(x.get('trip_id',''))[:8],
                            'fare_amount':x.get('fare_amount'), 'tip_amount':x.get('tip_amount'),
                            'ingest_batch_id':x.get('ingest_batch_id')})
        else:
            entries.append({'Mốc':label, 'Silver version':version,
                            'trip_id (8 ký tự)':trip_id[:8],
                            'fare_amount':'KHÔNG CÓ', 'tip_amount':'KHÔNG CÓ',
                            'ingest_batch_id':'—'})
    display(pd.DataFrame(entries))

print('Project:', ROOT)
print('Demo versions:', INITIAL_VERSION, UPDATE_VERSION, INSERT_VERSION, SCHEMA_VERSION)

Project: D:\delta-lakehouse-project
Demo versions: 44 45 47 48


## 1. DESCRIBE HISTORY — chứng minh các transaction được lưu

Delta version của riêng bảng Silver.

In [3]:
hist = DeltaTable.forPath(spark, SILVER).history()
important = (hist.filter(F.col('version').between(INITIAL_VERSION, SCHEMA_VERSION))
             .select('version','timestamp','operation','operationMetrics')
             .orderBy('version'))
display(important.show(truncate=False))
versions_present = {int(x['version']) for x in important.select('version').collect()}
print('Đủ tất cả mốc 44–48:', set(range(INITIAL_VERSION, SCHEMA_VERSION + 1)).issubset(versions_present))
print('Chú ý: Xem operation tại v46, không tự giả định đây là INSERT.')

+-------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                                                                                                                                    

None

Đủ tất cả mốc 44–48: True
Chú ý: Xem operation tại v46, không tự giả định đây là INSERT.


## 2. TIME TRAVEL chính — cùng một trip trước/sau UPDATE

Đây là trọng tâm Task 3. **Trước MERGE = version 44, sau MERGE UPDATE = version 45**. Đọc lại snapshot v44 *ngay cả khi Silver hiện tại đã lên v48*.

Lệnh cốt lõi phải trình chiếu:

```python
spark.read.format("delta").option("versionAsOf", 44).load(SILVER)
spark.read.format("delta").option("versionAsOf", 45).load(SILVER)
```

Kết quả kỳ vọng: cả hai version đều có cùng `trip_id`; ít nhất một trong `fare_amount`, `tip_amount` đã thay đổi.

In [4]:
past_silver = spark.read.format('delta').option('versionAsOf', INITIAL_VERSION).load(SILVER)
updated_silver = spark.read.format('delta').option('versionAsOf', UPDATE_VERSION).load(SILVER)

compare_cols = ['trip_id','fare_amount','tip_amount','ingest_batch_id']
past = (past_silver.filter(F.col('trip_id') == UPDATE_TRIP_ID)
        .select(*compare_cols).limit(2).collect())
new = (updated_silver.filter(F.col('trip_id') == UPDATE_TRIP_ID)
       .select(*compare_cols).limit(2).collect())
past = [r.asDict() for r in past]
new = [r.asDict() for r in new]
display_comparison(past, new, INITIAL_VERSION, UPDATE_VERSION, UPDATE_TRIP_ID)
if past and new:
    diff = pd.DataFrame([
        {'Trường':field,'BEFORE v44':past[0][field], 'AFTER v45':new[0][field],
         'Chênh lệch': (new[0][field] or 0) - (past[0][field] or 0)}
        for field in ('fare_amount','tip_amount')
    ])
    display(diff)
    print('TIME TRAVEL UPDATE:', 'PASS' if any(diff['Chênh lệch'] != 0)
          and new[0]['ingest_batch_id'] == UPDATE_INGEST_BATCH else 'KIỂM TRA LẠI')
else:
    print('THIẾU TRIP ở v44 hoặc v45. Kiểm tra ID / version / bảng.')

,Mốc,Silver version,trip_id (8 ký tự),fare_amount,tip_amount,ingest_batch_id
0,BEFORE,44,93184dd0,26.8,5.86,yellow_tripdata_2025-09
1,AFTER,45,93184dd0,28.8,6.86,demo_20260927T080446776977


,Trường,BEFORE v44,AFTER v45,Chênh lệch
0,fare_amount,26.80,28.80,2.0
1,tip_amount,5.86,6.86,1.0


TIME TRAVEL UPDATE: PASS


## 3. TIME TRAVEL phụ — INSERT và schema cũng có lịch sử

**INSERT:** v46 chưa có `new_trip_id` → v47 có record.

**SCHEMA:** v47 chưa có `surcharge_fee` → v48 có trường mới. Nếu v46 có một giao dịch khác, vẫn sử dụng v46 vì đây là version ngay trước v47.

In [5]:
old_insert, new_insert = trip_at(INSERT_VERSION - 1, INSERT_TRIP_ID), trip_at(INSERT_VERSION, INSERT_TRIP_ID)
print('INSERT | v46 BEFORE → v47 AFTER')
display_comparison(old_insert, new_insert, INSERT_VERSION-1, INSERT_VERSION, INSERT_TRIP_ID)
print('INSERT HISTORY:', 'PASS' if not old_insert and new_insert else 'KIỂM TRA LẠI')
old_schema, new_schema = at_version(INSERT_VERSION).schema, at_version(SCHEMA_VERSION).schema
old_names = set(old_schema.fieldNames())
new_names = set(new_schema.fieldNames())
display(pd.DataFrame([
    {'Snapshot':'BEFORE', 'Version':INSERT_VERSION, 'surcharge_fee': 'surcharge_fee' in old_names},
    {'Snapshot':'AFTER', 'Version':SCHEMA_VERSION, 'surcharge_fee': 'surcharge_fee' in new_names}
]))
print('SCHEMA HISTORY:', 'PASS' if 'surcharge_fee' not in old_names and 'surcharge_fee' in new_names
      else 'KIỂM TRA LẠI')

INSERT | v46 BEFORE → v47 AFTER


,Mốc,Silver version,trip_id (8 ký tự),fare_amount,tip_amount,ingest_batch_id
0,BEFORE,46,33f764b3,KHÔNG CÓ,KHÔNG CÓ,—
1,AFTER,47,33f764b3,28.8,6.86,demo_20260927T085807873071


INSERT HISTORY: PASS


,Snapshot,Version,surcharge_fee
0,BEFORE,47,False
1,AFTER,48,True


SCHEMA HISTORY: PASS


### 3.1. BEFORE → AFTER ở mức bản ghi khi Schema Evolution

Ở v47, `surcharge_fee` **chưa tồn tại trong schema**. Ở v48, cùng `UPDATE_TRIP_ID` được truy vấn lại, có thêm `surcharge_fee` (dữ liệu demo mong đợi: `1.5`). Cell xử lý cột không có ở v47 bằng nhãn `CHƯA CÓ CỘT`, không cố `select` một cột chưa tồn tại.

> Đây là Time Travel đọc **hai snapshot lịch sử**, không đọc riêng một DataFrame Silver hiện tại hai lần.


In [6]:
# Time Travel ở mức bản ghi: cùng trip_id, trước và sau schema evolution.
schema_before_rows = trip_at(INSERT_VERSION, UPDATE_TRIP_ID)  # v47
schema_after_rows = trip_at(SCHEMA_VERSION, UPDATE_TRIP_ID)  # v48

schema_comparison = []
for label, version, rows, has_column in [
    ('BEFORE SCHEMA', INSERT_VERSION, schema_before_rows, 'surcharge_fee' in old_names),
    ('AFTER SCHEMA', SCHEMA_VERSION, schema_after_rows, 'surcharge_fee' in new_names),
]:
    record = rows[0] if rows else {}
    schema_comparison.append({
        'Mốc': label,
        'Silver version': version,
        'trip_id (8 ký tự)': UPDATE_TRIP_ID[:8],
        'Có bản ghi': bool(rows),
        'fare_amount': record.get('fare_amount'),
        'tip_amount': record.get('tip_amount'),
        'surcharge_fee': (
            record.get('surcharge_fee') if has_column else 'CHƯA CÓ CỘT'
        ),
        'ingest_batch_id': record.get('ingest_batch_id'),
    })

display(pd.DataFrame(schema_comparison))

value_after = (schema_after_rows[0].get('surcharge_fee')
               if schema_after_rows else None)
schema_demo_pass = (
    bool(schema_before_rows) and bool(schema_after_rows)
    and 'surcharge_fee' not in old_names
    and 'surcharge_fee' in new_names
    and value_after is not None and float(value_after) == 1.5
)
print('SCHEMA BEFORE/AFTER RECORD:', 'PASS' if schema_demo_pass else 'KIỂM TRA LẠI')
if not schema_after_rows:
    print('Không tìm thấy trip demo ở v48; hãy đối chiếu trip_id và bản ghi schema CDC.')


,Mốc,Silver version,trip_id (8 ký tự),Có bản ghi,fare_amount,tip_amount,surcharge_fee,ingest_batch_id
0,BEFORE SCHEMA,47,93184dd0,True,28.8,6.86,CHƯA CÓ CỘT,demo_20260927T080446776977
1,AFTER SCHEMA,48,93184dd0,True,30.8,7.86,1.5,demo_20260927T091027666122


SCHEMA BEFORE/AFTER RECORD: PASS


## 4. AUDIT VERIFICATION — đối chiếu microbatch và Silver version

`batch_audit` là bảng pipeline tự ghi (incoming/valid/rejected/winners/version trước–sau); `DESCRIBE HISTORY` là lịch sử **giao dịch Delta**. Hai nguồn bằng chứng này bổ sung nhau nhưng **không phải một transaction nguyên tử xuyên nhiều bảng**.

Chỉ lọc các mốc CDC để trình chiếu gọn; **nếu v46 cũng xuất hiện, xem batch và operation thực tế của nó**.

In [8]:
audit = spark.read.format('delta').load(AUDIT)
audit_cols = [c for c in ('batch_id','incoming','valid','rejected','winners',
                          'silver_version_before','silver_version_after','silver_operation',
                          'silver_operation_metrics') if c in audit.columns]
cdc_audit = (audit.filter(F.col('silver_version_after').isin(45,46,47,48))
             .select(*audit_cols).orderBy('silver_version_after'))
display(cdc_audit.show(truncate=False))
print('Đối chiếu: silver_version_after trong audit ↔ version của history bên trên.')
print('Đừng thay version 46 bằng 47: chúng là hai commit khác nhau.')

+--------+--------+-----+--------+-------+---------------------+--------------------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|batch_id|incoming|valid|rejected|winners|silver_version_before|silver_version_after|silver_operation|silver_operation_metrics                                                                                                                                                                                                   

None

Đối chiếu: silver_version_after trong audit ↔ version của history bên trên.
Đừng thay version 46 bằng 47: chúng là hai commit khác nhau.


## 5. `_delta_log/*.json` — chứng minh Delta quản lý snapshot thế nào

Giảng viên yêu cầu **mở transaction logs thật**, không chỉ export file JSON báo cáo. Đọc mỗi file commit v44, 45, 46, 47, 48 và chỉ trình bày thông tin cốt lõi để không in hàng nghìn actions:

- `commitInfo.operation`: thao tác nào tạo version.
- `add` / `remove`: Delta đánh dấu file Parquet thêm/loại khỏi snapshot.
- `metaData.schemaString`: schema thay đổi khi evolution (nếu commit có metadata).

**Lưu ý:** Không tự khẳng định một MERGE thay đúng một file; số add/remove cần đọc từ log thực tế.

In [9]:
import json
log_dir = Path(SILVER) / '_delta_log'
result = []
for version in range(INITIAL_VERSION, SCHEMA_VERSION + 1):
    commit = log_dir / f'{version:020d}.json'
    if not commit.exists():
        result.append({'version': version, 'status': 'KHÔNG THẤY JSON',
                       'operation': None, 'add_files': None, 'remove_files': None, 'metadata_columns': None})
        continue
    counts = {'add': 0, 'remove': 0}
    info, meta_cols = {}, None
    with commit.open(encoding='utf-8') as fh:
        for line in fh:
            action = json.loads(line)
            if 'add' in action: counts['add'] += 1
            if 'remove' in action: counts['remove'] += 1
            if 'commitInfo' in action: info = action['commitInfo']
            if 'metaData' in action:
                schema_json = json.loads(action['metaData']['schemaString'])
                meta_cols = [field['name'] for field in schema_json.get('fields', [])]
    result.append({'version':version,'status':'OK','operation':info.get('operation'),
                   'add_files':counts['add'],'remove_files':counts['remove'],
                   'metadata_columns': ', '.join(meta_cols[-6:]) if meta_cols else '—'})
display(pd.DataFrame(result))
print('Với v48, tra cả schema của snapshot ở mục 3 để xác nhận surcharge_fee.')

,version,status,operation,add_files,remove_files,metadata_columns
0,44,OK,MERGE,16,16,—
1,45,OK,MERGE,1,1,—
2,46,OK,MERGE,1,0,—
3,47,OK,MERGE,1,0,—
4,48,OK,MERGE,1,1,"raw_record_hash, pickup_latitude, pickup_longi..."


Với v48, tra cả schema của snapshot ở mục 3 để xác nhận surcharge_fee.


In [10]:
# đóng spark
spark.stop()
print('Hoàn tất demo — không thay đổi Bronze, Silver, checkpoint hay _delta_log.')

Hoàn tất demo — không thay đổi Bronze, Silver, checkpoint hay _delta_log.
